Grad-CAM

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model        = model
        self.target_layer = target_layer
        self.gradients    = None
        self.activations  = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(_, __, output):
            self.activations = output.detach()

        def backward_hook(_, __, grad_output):
            self.gradients = grad_output[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()
        input_tensor = input_tensor.unsqueeze(0).to(device)
        input_tensor.requires_grad_()

        output     = self.model(input_tensor)
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        self.model.zero_grad()
        output[0, class_idx].backward()

        weights = self.gradients.mean(dim=[2, 3], keepdim=True)  # GAP over spatial dims
        cam     = (weights * self.activations).sum(dim=1).squeeze()
        cam     = torch.relu(cam)
        cam     = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()
        return cam.cpu().numpy(), class_idx


def overlay_heatmap(img_tensor, cam, alpha=0.45):
    """Overlay Grad-CAM heatmap on the original image."""
    # Denormalize
    mean = torch.tensor(NORM_MEAN).view(3, 1, 1)
    std  = torch.tensor(NORM_STD).view(3, 1, 1)
    img  = img_tensor.cpu() * std + mean
    img  = img.clamp(0, 1).permute(1, 2, 0).numpy()
    img  = (img * 255).astype(np.uint8)

    # Resize CAM to image size
    cam_resized = cv2.resize(cam, (img.shape[1], img.shape[0]))
    heatmap     = cv2.applyColorMap((cam_resized * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heatmap     = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    blended = (alpha * heatmap + (1 - alpha) * img).astype(np.uint8)
    return img, blended


def visualize_gradcam(model, dataset, gradcam, n_samples=6, split_name='Test', seed=42):
    np.random.seed(seed)
    indices = np.random.choice(len(dataset), n_samples, replace=False)

    fig, axes = plt.subplots(n_samples, 3, figsize=(11, n_samples * 3.2))
    fig.suptitle(f'Grad-CAM Heatmaps — {split_name}',
                 fontsize=13, fontweight='bold', y=0.995)

    for row, idx in enumerate(indices):
        img_tensor, label = dataset[idx]
        cam, pred_idx = gradcam.generate(img_tensor)

        original, overlay = overlay_heatmap(img_tensor, cam)
        pred_label = CLASS_NAMES[pred_idx]
        true_label = CLASS_NAMES[label]
        correct = pred_idx == label

        axes[row, 0].imshow(original)
        axes[row, 0].set_title(f'True: {true_label}', fontsize=8, pad=6)
        axes[row, 0].axis('off')

        axes[row, 1].imshow(cv2.resize(cam, (IMG_SIZE, IMG_SIZE)), cmap='jet')
        axes[row, 1].set_title('CAM (raw)', fontsize=8, pad=6)
        axes[row, 1].axis('off')

        border_color = '#2e7d32' if correct else '#c62828'
        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title(
            f'Pred: {pred_label}',
            fontsize=8,
            color='green' if correct else 'red',
            pad=6
        )
        for spine in axes[row, 2].spines.values():
            spine.set_edgecolor(border_color)
            spine.set_linewidth(2)
        axes[row, 2].axis('off')

    plt.tight_layout(rect=[0, 0, 1, 0.96])   # reserve space for suptitle
    out_path = os.path.join(SAVE_DIR, f'gradcam_{split_name.replace(" ","_")}.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Saved to: {out_path}")
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f'gradcam_{split_name.replace(" ","_")}.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Saved to: {SAVE_DIR}/gradcam_{split_name}.png")


# Attach Grad-CAM to last conv block of ResNet-50
gradcam = GradCAM(model, model.layer4[-1])

print("Generating Grad-CAM for Test_ID (in-distribution)...")
visualize_gradcam(model, test_id_dataset, gradcam,
                  n_samples=6, split_name='Test_ID')

print("\nGenerating Grad-CAM for Test_OOD (out-of-distribution)...")
visualize_gradcam(model, test_ood_dataset, gradcam,
                  n_samples=6, split_name='Test_OOD')

Vanilla Saliency map
Unlike Grad-CAM (which highlights regions in the final feature map), a vanilla saliency map computes the gradient of the predicted class score with respect to each INPUT PIXEL.  The absolute magnitude tells us which pixels most influence the decision, giving fine-grained, pixel-level insight.

In [ ]:
def compute_saliency(model, img_tensor, class_idx=None):
    """Return a 2-D saliency map and the predicted class index."""
    model.eval()
    inp = img_tensor.unsqueeze(0).to(device)
    inp.requires_grad_()

    output    = model(inp)
    if class_idx is None:
        class_idx = output.argmax(dim=1).item()

    model.zero_grad()
    output[0, class_idx].backward()

    # Max absolute gradient across RGB channels → single spatial map
    saliency        = inp.grad.data.abs().squeeze()   # (3, H, W)
    saliency, _     = saliency.max(dim=0)             # (H, W)
    saliency        = saliency.cpu().numpy()

    # Normalise to [0, 1]
    saliency = saliency - saliency.min()
    if saliency.max() > 0:
        saliency = saliency / saliency.max()

    return saliency, class_idx


def overlay_saliency(img_tensor, saliency, alpha=0.50):
    """Blend a saliency heatmap onto the original image."""
    mean = torch.tensor(NORM_MEAN).view(3, 1, 1)
    std  = torch.tensor(NORM_STD).view(3, 1, 1)
    img  = img_tensor.cpu() * std + mean
    img  = img.clamp(0, 1).permute(1, 2, 0).numpy()
    img  = (img * 255).astype(np.uint8)

    sal_resized = cv2.resize(saliency, (img.shape[1], img.shape[0]))
    heatmap     = cv2.applyColorMap((sal_resized * 255).astype(np.uint8),
                                    cv2.COLORMAP_HOT)          # HOT palette distinguishes from Grad-CAM's JET
    heatmap     = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    blended = (alpha * heatmap + (1 - alpha) * img).astype(np.uint8)
    return img, blended


def visualize_saliency(model, dataset, n_samples=6, split_name='Test', seed=42):
    np.random.seed(seed)
    indices = np.random.choice(len(dataset), n_samples, replace=False)

    fig, axes = plt.subplots(n_samples, 3, figsize=(11, n_samples * 3.2))
    fig.suptitle(f'Vanilla Saliency Maps — {split_name}',
                 fontsize=13, fontweight='bold', y=0.995)

    for row, idx in enumerate(indices):
        img_tensor, label = dataset[idx]
        saliency, pred_idx = compute_saliency(model, img_tensor)

        original, overlay  = overlay_saliency(img_tensor, saliency)
        pred_label  = CLASS_NAMES[pred_idx]
        true_label  = CLASS_NAMES[label]
        correct     = pred_idx == label

        # Column 0 — original image
        axes[row, 0].imshow(original)
        axes[row, 0].set_title(f'True: {true_label}', fontsize=8, pad=6)
        axes[row, 0].axis('off')

        # Column 1 — raw saliency map (grayscale)
        axes[row, 1].imshow(cv2.resize(saliency, (IMG_SIZE, IMG_SIZE)), cmap='hot')
        axes[row, 1].set_title('Saliency (raw)', fontsize=8, pad=6)
        axes[row, 1].axis('off')

        # Column 2 — blended overlay with correct/incorrect border
        border_color = '#2e7d32' if correct else '#c62828'
        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title(
            f'Pred: {pred_label}',
            fontsize=8,
            color='green' if correct else 'red',
            pad=6
        )
        for spine in axes[row, 2].spines.values():
            spine.set_edgecolor(border_color)
            spine.set_linewidth(2)
        axes[row, 2].axis('off')

    plt.tight_layout(rect=[0, 0, 1, 0.96])   # reserve space for suptitle
    out_path = os.path.join(SAVE_DIR, f'saliency_{split_name.replace(" ", "_")}.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Saved to: {out_path}")
    plt.close()


print("Generating Saliency Maps for Test_ID (in-distribution)...")
visualize_saliency(model, test_id_dataset,
                   n_samples=6, split_name='Test_ID')

print("\nGenerating Saliency Maps for Test_OOD (out-of-distribution)...")
visualize_saliency(model, test_ood_dataset,
                   n_samples=6, split_name='Test_OOD')